# AIE425 — Intelligent Recommender Systems — Lab 3

Complete solution. Run top-to-bottom. The notebook creates the three CSVs, metrics, plots, missing-data experiment, and cold-start measurements.

In [ ]:

import random, math, csv, statistics
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

random.seed(425)

# Part 2: samplers; all distribution draws use random.random().
def sample_uniform(a, b):
    return a + (b-a) * random.random()

def sample_normal(mu, sigma):
    u1 = 1 - random.random()  # avoids log(0)
    u2 = random.random()
    z = math.sqrt(-2 * math.log(u1)) * math.cos(2 * math.pi * u2)
    return mu + sigma*z

def sample_poisson(lam):
    L, k, p = math.exp(-lam), 0, 1.0
    while True:
        p *= random.random()
        if p <= L:
            return k
        k += 1

# Part 3: generate integer ratings on the 1–5 scale.
def generate_ratings(n, dist, **params):
    result = []
    for _ in range(n):
        if dist == "uniform":
            raw = sample_uniform(0.5, 5.5)
        elif dist == "normal":
            raw = sample_normal(params.get("mu", 3), params.get("sigma", 1))
        elif dist == "poisson":
            raw = 1 + sample_poisson(params.get("lam", 2))
        else:
            raise ValueError("dist must be uniform, normal, or poisson")
        result.append(max(1, min(5, round(raw))))
    return result

def assign_ratings(n_items, dist, max_fill=0.5, **params):
    row = [None]*n_items
    k = random.randint(5, int(max_fill*n_items))
    chosen = random.sample(range(n_items), k)
    ratings = generate_ratings(k, dist, **params)
    for i, r in zip(chosen, ratings):
        row[i] = r
    assert sum(r is not None for r in row) <= 0.5*n_items
    return row

def build_matrix(dist, n_users=5000, n_items=100, **params):
    return [assign_ratings(n_items, dist, **params) for _ in range(n_users)]

def save_interaction_csv(matrix, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["user_id","item_id","rating"])
        for u,row in enumerate(matrix):
            for i,r in enumerate(row):
                if r is not None: w.writerow([u,i,r])

def observed(matrix):
    return [r for row in matrix for r in row if r is not None]

def sparsity(matrix):
    return 1 - sum(r is not None for row in matrix for r in row)/(len(matrix)*len(matrix[0]))

def global_mean(matrix):
    vals = observed(matrix)
    return sum(vals)/len(vals) if vals else float("nan")

def user_means(matrix):
    out = []
    for row in matrix:
        vals = [r for r in row if r is not None]
        if vals: out.append(sum(vals)/len(vals))
    return out

def mean_of_user_means(matrix):
    vals = user_means(matrix)
    return sum(vals)/len(vals)

def item_means(matrix):
    out=[]
    for i in range(len(matrix[0])):
        vals=[row[i] for row in matrix if row[i] is not None]
        if vals: out.append(sum(vals)/len(vals))
    return out

def normalized_entropy(matrix):
    counts = Counter(observed(matrix)); n=sum(counts.values())
    ps=[counts.get(v,0)/n for v in range(1,6)]
    return -sum(p*math.log(p) for p in ps if p>0)/math.log(5)

def chi2_uniform(matrix):
    counts=Counter(observed(matrix)); n=sum(counts.values()); e=n/5
    return sum((counts.get(v,0)-e)**2/e for v in range(1,6))

def gini(values):
    xs=sorted(values); n=len(xs); total=sum(xs)
    if n==0 or total==0: return 0.0
    return 2*sum((i+1)*x for i,x in enumerate(xs))/(n*total)-(n+1)/n

def homogeneity_metrics(matrix):
    pop=[sum(row[i] is not None for row in matrix) for i in range(len(matrix[0]))]
    ums=user_means(matrix); ims=item_means(matrix)
    return {
        "Sparsity": sparsity(matrix),
        "Global mean": global_mean(matrix),
        "Mean of user means": mean_of_user_means(matrix),
        "Smallest / largest user mean": (min(ums),max(ums)),
        "Normalized entropy": normalized_entropy(matrix),
        "Chi-square vs uniform": chi2_uniform(matrix),
        "Std of user means": statistics.pstdev(ums),
        "Std of item means": statistics.pstdev(ims),
        "Gini of item popularity": gini(pop)
    }

def complete_matrix(n_users,n_items,dist,**params):
    return [generate_ratings(n_items,dist,**params) for _ in range(n_users)]

def hide_ratings(full,p,mechanism):
    # Hides exactly p fraction, rounded to the nearest cell.
    n_users,n_items=len(full),len(full[0]); N=n_users*n_items
    n_keep=round((1-p)*N)
    if mechanism=="random":
        kept=set(random.sample(range(N),n_keep))
        return [[full[u][i] if u*n_items+i in kept else None for i in range(n_items)] for u in range(n_users)]
    if mechanism=="dislike":
        # Key specified by the lab: u / rating. Keep largest keys.
        # NumPy is used as a storage/selection array for speed; random.random()
        # remains the only source of random values.
        keys=np.fromiter((random.random()/full[u][i] for u in range(n_users) for i in range(n_items)),dtype=float,count=N)
        keep_idx=np.argpartition(keys, N-n_keep)[N-n_keep:]
        keep_mask=np.zeros(N,dtype=bool); keep_mask[keep_idx]=True
        return [[full[u][i] if keep_mask[u*n_items+i] else None for i in range(n_items)] for u in range(n_users)]
    raise ValueError("mechanism must be random or dislike")

def add_cold_start(matrix,dist):
    old_users,old_items=len(matrix),len(matrix[0])
    result=[row[:] + [None]*10 for row in matrix]
    for j in range(old_items,old_items+10):
        k=random.randint(0,2)
        users=random.sample(range(old_users),k)
        for u,r in zip(users,generate_ratings(k,dist)): result[u][j]=r
    for _ in range(500):
        row=[None]*(old_items+10); k=random.randint(1,3)
        items=random.sample(range(old_items),k)
        for i,r in zip(items,generate_ratings(k,dist)): row[i]=r
        result.append(row)
    return result

def rated_item_sets(matrix):
    return [set(i for i,r in enumerate(row) if r is not None) for row in matrix]

def cold_start_metrics(before,after,n_old_users=5000,n_old_items=100):
    cold_sets=rated_item_sets(after[n_old_users:])
    old_sets=rated_item_sets(after[:n_old_users])
    cold_item_counts=[sum(row[i] is not None for row in after) for i in range(n_old_items,len(after[0]))]
    # Exact over all 500 x 5000 cold-old pairs, using set intersections.
    overlaps=[len(a & b) for a in cold_sets for b in old_sets]
    old_pairs=[(random.sample(range(n_old_users),2)) for _ in range(1000)]
    old_old=sum(len(old_sets[a] & old_sets[b]) for a,b in old_pairs)/1000
    return {
        "Sparsity before":sparsity(before),
        "Sparsity after":sparsity(after),
        "Cold users (<5 ratings)":sum(len(s)<5 for s in cold_sets),
        "Cold items (<5 ratings)":sum(c<5 for c in cold_item_counts),
        "Mean co-rated items (cold-old)":sum(overlaps)/len(overlaps),
        "Mean co-rated items (old-old, 1000 pairs)":old_old,
        "No-overlap cold-old pairs (%)":100*sum(x==0 for x in overlaps)/len(overlaps)
    }


## Part 2 — Sampler validation

In [ ]:
random.seed(425)
sampler_stats={}
for name,fn in [("Uniform U(1,5)",lambda:sample_uniform(1,5)),("Normal N(3,1)",lambda:sample_normal(3,1)),("Poisson lambda=2",lambda:sample_poisson(2))]:
    xs=[fn() for _ in range(100000)]
    sampler_stats[name]=(sum(xs)/len(xs),statistics.pvariance(xs))
sampler_stats

## Part 3 — Generate matrices and export CSVs

In [ ]:
random.seed(425)
matrices={}
for d in ["uniform","normal","poisson"]:
    matrices[d]=build_matrix(d)
    save_interaction_csv(matrices[d],f"{d}_ratings.csv")
{d:(len(m),len(m[0]),sum(x is not None for row in m for x in row)) for d,m in matrices.items()}

## Part 4 — Metrics and plots

In [ ]:
metrics={d:homogeneity_metrics(m) for d,m in matrices.items()}
for d,vals in metrics.items():
    print("\\n"+d.upper())
    for k,v in vals.items(): print(f"{k}: {v}")

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4),sharey=True)
for ax,d in zip(axes,["uniform","normal","poisson"]):
    ax.hist(observed(matrices[d]),bins=np.arange(.5,6.5,1),rwidth=.85)
    ax.set_title(d.capitalize()); ax.set_xticks([1,2,3,4,5]); ax.set_xlabel("Rating")
axes[0].set_ylabel("Count"); fig.tight_layout(); plt.show()

fig,axes=plt.subplots(1,3,figsize=(15,5),sharey=True)
for ax,d in zip(axes,["uniform","normal","poisson"]):
    arr=np.array([[np.nan if x is None else x for x in row] for row in matrices[d][:50]],float)
    cmap=plt.cm.viridis.copy(); cmap.set_bad("white")
    im=ax.imshow(np.ma.masked_invalid(arr),aspect="auto",vmin=1,vmax=5,cmap=cmap)
    ax.set_title(d.capitalize()); ax.set_xlabel("Item")
axes[0].set_ylabel("First 50 users"); fig.colorbar(im,ax=axes,label="Rating",shrink=.8); plt.show()

## Part 5 — Missing-data experiment (20 trials per condition)

In [ ]:
random.seed(425)
full=complete_matrix(5000,100,"normal",mu=3,sigma=1)
ref_mu=global_mean(full); ref_user_mean=mean_of_user_means(full)
experiment_rows=[]
for mechanism in ["random","dislike"]:
    for p in [.50,.75,.90]:
        em,eu=[],[]
        for _ in range(20):
            hidden=hide_ratings(full,p,mechanism)
            em.append(global_mean(hidden)-ref_mu)
            eu.append(mean_of_user_means(hidden)-ref_user_mean)
        experiment_rows.append({"Mechanism":mechanism,"Hidden":f"{int(p*100)}%",
            "Bias of global mean":statistics.mean(em),"Random error of global mean":statistics.pstdev(em),
            "Bias of mean user means":statistics.mean(eu),"Random error of mean user means":statistics.pstdev(eu)})
print("Reference means:",ref_mu,ref_user_mean)
for row in experiment_rows: print(row)

## Part 6 — Cold start

In [ ]:
random.seed(425)
cold_after=add_cold_start(matrices["uniform"],"uniform")
cold_metrics=cold_start_metrics(matrices["uniform"],cold_after)
cold_metrics

## Part 7 — Discussion answers

1. Uniform should have the most even histogram, though rounding and clipping make it approximate.
2. Sparsity is driven by the number of items rated per user, not the rating distribution.
3. Rating 3 should be most common in the normal matrix because its mean is 3.
4. Use the measured cold-old overlap in `cold_metrics`. Cosine similarity is undefined when there are no co-rated items.
5. Asking new users to choose liked titles addresses new-user cold start.
6. The global mean weights ratings; the mean of user means weights users. They can differ when users have different rating counts and means.
7. Random hiding should have near-zero bias. The lab's literal `key = u / rating`, keep-largest rule actually retains low ratings more often and biases observed averages downward, contradicting the prose description. To hide low ratings more often, reverse the selection direction or use a rating-proportional keep key. Random error generally increases as more ratings are hidden.

## Submission checklist

Submit `Lab3.ipynb` and the three CSV files in `Activity_3/`, with plots and filled tables, then commit and push.